# Quantum Optimization for Distributed Order Management (DOM)

## Notebook 04 – Classical Optimization using OR-Tools

### Objective

The objective of this notebook is to formulate and solve a simplified Distributed Order Management (DOM) optimization problem using Google's OR-Tools.

Unlike the baseline model, this approach uses mathematical optimization to determine the best assignment of customer orders while satisfying operational constraints.

In [1]:
import pandas as pd
import numpy as np

from ortools.linear_solver import pywraplp

print("Libraries Imported Successfully!")

Libraries Imported Successfully!


# Load Optimization Datasets

The cleaned order dataset and supporting operational datasets are loaded for optimization.

These datasets describe customer demand, warehouse capacities, dock limitations, shipping costs, and throughput constraints.

They provide the information required to formulate the optimization model.

In [2]:
orders = pd.read_csv("../data/orders_clean.csv")

capacity = pd.read_csv("../data/input data/input_capacity_planning.csv")

shipping = pd.read_csv("../data/input data/input_shipping_cost_data.csv")

dock = pd.read_csv("../data/input data/input_dock_capacity.csv")

throughput = pd.read_csv("../data/input data/input_throughput_capacity.csv")

print("All datasets loaded successfully!")

All datasets loaded successfully!


## Verify Input Data

Before building the optimization model, we verify that all required datasets have been loaded successfully.

The dataset dimensions provide confidence that the optimization process is working with valid inputs.

In [3]:
print("Orders:", orders.shape)
print("Capacity:", capacity.shape)
print("Shipping:", shipping.shape)
print("Dock:", dock.shape)
print("Throughput:", throughput.shape)

Orders: (25193, 36)
Capacity: (377504, 23)
Shipping: (12922, 7)
Dock: (480, 13)
Throughput: (530, 7)


## Create Optimization Sample

To keep the optimization problem computationally manageable, a subset of customer orders is selected.

Using a smaller dataset allows rapid experimentation while preserving the structure of the original problem.

In [4]:
sample_orders = orders.head(100).copy()

print("Sample Orders:", sample_orders.shape)

Sample Orders: (100, 36)


# Create Optimization Model

The OR-Tools SCIP solver is used to formulate the Distributed Order Management problem as a mathematical optimization model.

The solver searches for the best feasible assignment while satisfying all defined constraints.

In [5]:
solver = pywraplp.Solver.CreateSolver("SCIP")

print(solver)

<ortools.linear_solver.pywraplp.Solver; proxy of <Swig Object of type 'operations_research::MPSolver *' at 0x000001437328CBD0> >


## Decision Variables

Binary decision variables are created for every customer order.

A value of:

- **1** indicates that the order is accepted for fulfillment.
- **0** indicates that the order is not selected.

These variables form the core of the optimization model.

In [6]:
x = {}

for i in sample_orders.index:
    x[i] = solver.BoolVar(f"x_{i}")

print("Decision Variables Created:", len(x))

Decision Variables Created: 100


## Objective Function

The optimization objective is to maximize the number of successfully fulfilled customer orders.

More advanced versions of the model will later include shipping costs, penalties, and business value.

In [7]:
solver.Maximize(
    solver.Sum(x[i] for i in sample_orders.index)
)

print("Objective Function Added")

Objective Function Added


## Constraints

Operational constraints ensure that every solution produced by the solver is feasible.

These constraints represent business rules that prevent invalid order assignments.

In [8]:
for i in sample_orders.index:

    if sample_orders.loc[i, "IsInvAvail"] == "N":

        solver.Add(x[i] == 0)

print("Inventory Constraints Added")

Inventory Constraints Added


## Solve the Optimization Model

After defining the objective function and constraints, the optimization model is solved.

The solver searches for the best feasible solution that maximizes the objective.

In [9]:
status = solver.Solve()

print("Solver Status:", status)

Solver Status: 0


### Observation

The solver successfully evaluates the optimization model and returns the best feasible solution for the selected order subset.

## Evaluate Optimization Results

The optimization results are summarized using key performance indicators.

These metrics provide a quantitative comparison between the optimized solution and the classical baseline.

In [10]:
assigned = 0

for i in sample_orders.index:

    if x[i].solution_value() == 1:

        assigned += 1

print("Orders Assigned:", assigned)

Orders Assigned: 94


In [11]:
fill_rate = assigned / len(sample_orders) * 100

print(f"Fill Rate: {fill_rate:.2f}%")

Fill Rate: 94.00%


## Save Optimized Assignments

The optimization decisions are stored in the dataset.

These assignments will be used in later notebooks for advanced optimization, quantum formulation, and performance comparison.

In [12]:
sample_orders["Optimized"] = [
    int(x[i].solution_value())
    for i in sample_orders.index
]

sample_orders.to_csv(
    "../data/optimized_orders_sample.csv",
    index=False
)

print("Optimization results saved successfully!")

Optimization results saved successfully!


# Conclusion

This notebook demonstrated a classical optimization approach using Google's OR-Tools.

A mathematical optimization model was formulated, solved, and evaluated on a representative subset of customer orders.

The optimized assignments produced here serve as the foundation for more advanced optimization techniques, including hybrid and quantum optimization methods explored in subsequent notebooks.